# Event Duration Bucket Analysis

Purpose: summarize raw event counts by duration bucket before applying pricing thresholds. This is a diagnostic notebook for the eventization / underwriter-confidence discussion; it does not feed pricing directly.

The key question: for each catalog, county, and threshold area, how much event mass sits in buckets such as `0-2h`, `2-4h`, `4-8h`, `8-12h`, `12-24h`, and `24h+`?

In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
CATALOGS = ["eagle-i-30min", "eagle-i-45min", "eagle-i-60min"]
DEFAULT_CATALOG = "eagle-i-45min"
OUTPUT_DIR = ROOT / "notebooks" / "outputs" / "event_duration_bucket_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BUCKET_EDGES = [0, 2, 4, 8, 12, 24, float("inf")]
BUCKET_LABELS = ["0-2h", "2-4h", "4-8h", "8-12h", "12-24h", "24h+"]
THRESHOLDS = [2, 4, 8, 12, 24]

def catalog_events(catalog_id: str) -> pd.DataFrame:
    path = ROOT / "price_engine" / "catalogs" / catalog_id / "data" / "events.parquet"
    cols = ["fips", "state", "county", "duration_hours", "max_customers", "mean_customers", "year"]
    df = pd.read_parquet(path, columns=cols)
    df["catalog_id"] = catalog_id
    df["duration_bucket"] = pd.cut(
        df["duration_hours"],
        bins=BUCKET_EDGES,
        labels=BUCKET_LABELS,
        right=False,
        include_lowest=True,
    )
    return df


## National Bucket Distribution

This table answers: before selecting a contract threshold, where do the raw event durations sit? The catalog ID controls the event-continuity tolerance: 30, 45, or 60 minutes.

In [ ]:
national_rows = []
for catalog_id in CATALOGS:
    df = catalog_events(catalog_id)
    total = len(df)
    grouped = df.groupby("duration_bucket", observed=False)
    for bucket, sub in grouped:
        national_rows.append({
            "catalog_id": catalog_id,
            "duration_bucket": str(bucket),
            "events": int(len(sub)),
            "event_share": len(sub) / total if total else 0.0,
            "mean_max_customers": sub["max_customers"].mean(),
            "median_max_customers": sub["max_customers"].median(),
            "p90_max_customers": sub["max_customers"].quantile(0.90),
        })

national_bucket_summary = pd.DataFrame(national_rows)
national_bucket_summary.to_csv(OUTPUT_DIR / "national_duration_bucket_summary.csv", index=False)
national_bucket_summary.style.format({
    "event_share": "{:.2%}",
    "mean_max_customers": "{:.1f}",
    "median_max_customers": "{:.1f}",
    "p90_max_customers": "{:.1f}",
})


## Default Catalog: Bucket And Cumulative Threshold View

The dashboard default is `eagle-i-45min`. Buckets are non-overlapping. Threshold counts are cumulative and match pricing-style questions: `duration_hours >= T`.

In [ ]:
default_events = catalog_events(DEFAULT_CATALOG)

def cumulative_threshold_counts(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    total = len(df)
    for T in THRESHOLDS:
        n = int((df["duration_hours"] >= T).sum())
        rows.append({
            "T_hours": T,
            "events_ge_T": n,
            "event_share_ge_T": n / total if total else 0.0,
        })
    return pd.DataFrame(rows)

def bucket_counts(df: pd.DataFrame) -> pd.DataFrame:
    counts = df["duration_bucket"].value_counts(sort=False)
    out = counts.rename_axis("duration_bucket").reset_index(name="events")
    out["event_share"] = out["events"] / len(df)
    return out

default_bucket_summary = bucket_counts(default_events)
default_threshold_summary = cumulative_threshold_counts(default_events)
default_bucket_summary.to_csv(OUTPUT_DIR / "default_45min_duration_bucket_summary.csv", index=False)
default_threshold_summary.to_csv(OUTPUT_DIR / "default_45min_cumulative_threshold_summary.csv", index=False)

display(default_bucket_summary.style.format({"event_share": "{:.2%}"}))
display(default_threshold_summary.style.format({"event_share_ge_T": "{:.2%}"}))


## County Examples

These examples are useful for discussion because they include Massachusetts launch counties, a known source-quality example, and a visual-review example.

In [ ]:
EXAMPLE_FIPS = [25027, 25025, 48095, 41025, 36029]

example_rows = []
for fips, sub in default_events[default_events["fips"].isin(EXAMPLE_FIPS)].groupby("fips"):
    name = f"{sub.iloc[0]['county']}, {sub.iloc[0]['state']}"
    counts = sub["duration_bucket"].value_counts(sort=False)
    row = {
        "fips": int(fips),
        "county_state": name,
        "total_events": int(len(sub)),
    }
    for label in BUCKET_LABELS:
        row[label] = int(counts.get(label, 0))
    for T in THRESHOLDS:
        row[f"T>={T}h"] = int((sub["duration_hours"] >= T).sum())
    example_rows.append(row)

example_county_bucket_summary = pd.DataFrame(example_rows).sort_values("fips")
example_county_bucket_summary.to_csv(OUTPUT_DIR / "example_county_duration_bucket_summary.csv", index=False)
example_county_bucket_summary


## County-Level Matrix Candidate

This optional output has one row per county and catalog, with raw duration buckets plus cumulative threshold counts. It can feed a future `confidence(fips, T)` artifact.

In [ ]:
county_rows = []
for catalog_id in CATALOGS:
    df = catalog_events(catalog_id)
    for (fips, state, county), sub in df.groupby(["fips", "state", "county"], sort=False):
        counts = sub["duration_bucket"].value_counts(sort=False)
        row = {
            "catalog_id": catalog_id,
            "fips": int(fips),
            "state": state,
            "county": county,
            "total_events": int(len(sub)),
        }
        for label in BUCKET_LABELS:
            row[f"bucket_{label}"] = int(counts.get(label, 0))
        for T in THRESHOLDS:
            row[f"events_ge_{T}h"] = int((sub["duration_hours"] >= T).sum())
        county_rows.append(row)

county_duration_bucket_matrix = pd.DataFrame(county_rows)
county_duration_bucket_matrix.to_csv(OUTPUT_DIR / "county_duration_bucket_matrix.csv", index=False)
county_duration_bucket_matrix.head()


## Interpretation

Use these outputs as diagnostics, not as direct pricing inputs.

- Buckets show raw event mass and help identify where threshold confidence should come from.
- Cumulative `T` counts show how many events remain for each pricing threshold.
- Comparing 30/45/60-minute catalogs shows whether the event distribution is sensitive to the continuity assumption.
- For a future confidence score, a cell with many `0-2h` events but unstable `T>=4h` counts should not be messaged the same way as a cell with stable `T>=8h` evidence.